# Ad Campaign CTR: Exploratory Analysis
Which platforms, channels, targeting types and budgets get more clicks per impression? Data: `Dataset.csv` (72,612 campaign-day rows, May to Dec 2022).

**Method:** clean the data, answer questions in SQL (SQLite), plot with Matplotlib/Seaborn. CTR is computed as total clicks / total impressions (weighted), not the average of row CTRs.

In [1]:
import sqlite3, re, warnings
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
TEAL, AMBER = '#0F4C5C', '#E9A23B'
Path('charts').mkdir(exist_ok=True); Path('results').mkdir(exist_ok=True)
raw = pd.read_csv('../Dataset.csv', low_memory=False)
con = sqlite3.connect(':memory:'); raw.to_sql('ads', con, index=False)
print(raw.shape)

(72612, 35)


## 1. Data quality

In [2]:
print('duplicate rows:', raw.duplicated().sum())
miss = raw.isna().mean().mul(100).round(1); print(miss[miss>0].to_string())
d = raw[raw.impressions>0].copy(); d['ctr'] = d.clicks/d.impressions
bad = d[d.ctr>0.20]
print(f'\nrows with CTR > 20% (impossible-looking, clicks close to or above impressions): {len(bad)}  | CTR > 100%: {(d.ctr>1).sum()}')
v = d[d.ctr<=0.20].copy()
print('valid rows kept:', len(v))

duplicate rows: 0
creative_width           4.7
creative_height          4.7
template_id              4.7
approved_budget          0.6
max_bid_cpm             89.8
position_in_content    100.0
unique_reach           100.0
total_reach            100.0

rows with CTR > 20% (impossible-looking, clicks close to or above impressions): 157  | CTR > 100%: 9


valid rows kept: 72455


`position_in_content`, `unique_reach` and `total_reach` are 100% empty and `max_bid_cpm` is ~90% empty, so none can be analysed. About 4.7% of rows have no creative size. 157 rows have a CTR above 20% and are treated as reporting glitches and excluded.

In [3]:
def run_sql(path='sql_queries.sql'):
    out = {}
    text = Path(path).read_text()
    for block in re.split(r'-- name: ', text)[1:]:
        name, _, sql = block.partition('\n')
        df = pd.read_sql(sql.strip().rstrip(';'), con)
        df.to_csv(f'results/{name.strip()}.csv', index=False); out[name.strip()] = df
    return out
Q = run_sql(); print(len(Q), 'queries executed')
Q['q01_data_quality']

12 queries executed


,total_rows,zero_impressions,ctr_over_20pct,missing_creative_size,missing_max_bid
0,72612,0,157,3412,65206


## 2. Headline CTR: weighted vs simple average

In [4]:
Q['q02_overall_ctr']

,valid_rows,impressions,clicks,weighted_ctr_pct,simple_avg_ctr_pct
0,72455,98256517,3167133,3.223,1.64


The two numbers differ by about 2x. The weighted CTR is dominated by big, high-volume campaign days, which click far more than the many small rows. **Using the simple average would understate real performance.**

In [5]:
r = Q['q02_overall_ctr'].iloc[0]
fig, ax = plt.subplots(figsize=(5.5,3.6))
bars = ax.bar(['Simple average\nof row CTRs','Weighted\n(clicks / impressions)'], [r.simple_avg_ctr_pct, r.weighted_ctr_pct], color=[AMBER, TEAL])
ax.bar_label(bars, fmt='%.2f%%'); ax.set_ylabel('CTR (%)'); ax.set_title('Same data, two definitions of CTR')
plt.tight_layout(); plt.savefig('charts/01_weighted_vs_simple_ctr.png', dpi=150); plt.show()

## 3. Platform and channel

In [6]:
fig, axes = plt.subplots(1,2, figsize=(11,3.8), sharey=False)
for ax,(k,xc,t) in zip(axes,[('q03_ctr_by_platform','platform','By ad platform'),('q04_ctr_by_channel','channel','By channel')]):
    df=Q[k]; sns.barplot(data=df,x=xc,y='ctr_pct',color=TEAL,ax=ax); ax.bar_label(ax.containers[0],fmt='%.2f'); ax.set_title(t); ax.set_ylabel('CTR (%)'); ax.set_xlabel('')
    ax.set_ylim(0, df.ctr_pct.max()*1.25)
plt.tight_layout(); plt.savefig('charts/02_platform_channel_ctr.png', dpi=150); plt.show()
Q['q03_ctr_by_platform']

,platform,rows_,impressions,clicks,ctr_pct
0,DV360,24121,33301735,1109480,3.332
1,Facebook Ads,24226,33057250,1072918,3.246
2,Google Ads,24108,31897532,984735,3.087


Platform and channel make little difference: all channels sit within about 0.13 percentage points of each other (3.17 to 3.30%). Where the ad runs matters far less than how it is targeted and who runs it.

## 4. Targeting type, advertiser, timezone

In [7]:
fig, axes = plt.subplots(1,2, figsize=(12,4.2))
t=Q['q05_ctr_by_targeting_type']; sns.barplot(data=t,y='targeting',x='ctr_pct',color=TEAL,ax=axes[0]); axes[0].bar_label(axes[0].containers[0],fmt='%.2f'); axes[0].set_title('CTR by targeting type'); axes[0].set_xlabel('CTR (%)'); axes[0].set_ylabel('')
a=Q['q06_top_advertisers']; sns.barplot(data=a,y='advertiser_name',x='ctr_pct',color=AMBER,ax=axes[1]); axes[1].bar_label(axes[1].containers[0],fmt='%.2f'); axes[1].set_title('Top 10 advertisers (>=100 rows)'); axes[1].set_xlabel('CTR (%)'); axes[1].set_ylabel('')
plt.tight_layout(); plt.savefig('charts/03_targeting_advertiser_ctr.png', dpi=150); plt.show()
t

,targeting,rows_,impressions,ctr_pct
0,Retargeting,2540,7307403,3.608
1,Others,48851,76100542,3.540
2,Youtube,7460,5090261,1.506
3,Inmarket,13016,9345297,1.373
4,Affinity,588,413014,1.194


Retargeting and Others get about 3.5 to 3.6% CTR, versus 1.2 to 1.5% for Youtube, Inmarket and Affinity targeting. Advertisers vary even more: the best (with 100+ rows) reach 4.5 to 7.3% while the data average is 3.2%.

## 5. Budget and campaign age

In [8]:
fig, axes = plt.subplots(1,2, figsize=(11,3.8))
b=Q['q07_ctr_by_budget_quintile']; lab=[f'Q{int(q)}\n${int(lo):,}-{int(hi):,}' for q,lo,hi in zip(b.budget_quintile,b.min_budget_usd,b.max_budget_usd)]
axes[0].bar(lab,b.ctr_pct,color=TEAL); axes[0].bar_label(axes[0].containers[0],fmt='%.2f'); axes[0].set_title('CTR by campaign budget quintile'); axes[0].set_ylabel('CTR (%)'); axes[0].tick_params(axis='x',labelsize=8)
c=Q['q09_campaign_length_buckets']; axes[1].bar(c.campaign_day_bucket,c.ctr_pct,color=AMBER); axes[1].bar_label(axes[1].containers[0],fmt='%.2f'); axes[1].set_title('CTR by campaign day'); axes[1].set_xlabel('Day of campaign'); axes[1].set_ylabel('CTR (%)')
plt.tight_layout(); plt.savefig('charts/04_budget_and_campaign_day.png', dpi=150); plt.show()
b

,budget_quintile,min_budget_usd,max_budget_usd,ctr_pct
0,1,2.0,151.0,1.431
1,2,151.0,325.0,1.737
2,3,325.0,472.0,1.963
3,4,472.0,761.0,2.536
4,5,761.0,39560.0,4.336


CTR climbs steadily with budget: the top quintile of budgets gets about 3x the CTR of the bottom (4.3% vs 1.4%). This is a **correlation, not proof that spending more causes more clicks**: large budgets may belong to advertisers or targeting types that already perform well. CTR peaks around days 8 to 14 (3.7%) and falls by more than half after day 30 (1.7%).

## 6. Over time

In [9]:
m=Q['q08_monthly_trend']
fig, ax = plt.subplots(figsize=(8,3.6)); ax.plot(m.month,m.ctr_pct,marker='o',color=TEAL); ax.set_title('Monthly CTR, 2022'); ax.set_ylabel('CTR (%)')
for x,y,n in zip(m.month,m.ctr_pct,m.rows_): ax.annotate(f'n={n:,}',(x,y),textcoords='offset points',xytext=(0,8),ha='center',fontsize=7)
ax.set_ylim(0,m.ctr_pct.max()*1.2); plt.tight_layout(); plt.savefig('charts/05_monthly_ctr.png', dpi=150); plt.show(); m

,month,rows_,impressions,clicks,ctr_pct
0,2022-05,481,2144911,81646,3.806
1,2022-06,4483,4205237,83048,1.975
2,2022-07,8024,11086497,258808,2.334
3,2022-08,8924,21323846,706235,3.312
4,2022-09,9191,18351272,692396,3.773
5,2022-10,20812,27667613,1119933,4.048
6,2022-11,15374,9818846,159867,1.628
7,2022-12,5166,3658295,65200,1.782


May has only 481 rows, so its figure is less reliable. After a peak in October (4.05%), CTR fell to 1.6 to 1.8% in November and December. Seasonality cannot be separated from a change in campaign mix with only one year of data.

## 7. Distribution of row-level CTR and creative size

In [10]:
fig, axes = plt.subplots(1,2, figsize=(11,3.8))
sns.histplot(v.ctr*100, bins=60, color=TEAL, ax=axes[0]); axes[0].axvline(v.ctr.median()*100,color=AMBER,ls='--',label=f'median {v.ctr.median()*100:.2f}%'); axes[0].legend(); axes[0].set_xlabel('Row-level CTR (%)'); axes[0].set_title('Distribution of daily CTR (valid rows)')
s=Q['q10_creative_size']; sns.barplot(data=s,x='creative_size',y='ctr_pct',color=AMBER,ax=axes[1]); axes[1].bar_label(axes[1].containers[0],fmt='%.2f'); axes[1].set_title('CTR by creative size'); axes[1].set_xlabel(''); axes[1].set_ylabel('CTR (%)'); axes[1].tick_params(axis='x',labelsize=8)
plt.tight_layout(); plt.savefig('charts/06_ctr_distribution_creative_size.png', dpi=150); plt.show(); s

,creative_size,rows_,ctr_pct
0,300x250,61716,2.686
1,0x0 (no size recorded),7425,1.709
2,missing,3314,5.586


Daily CTR is right-skewed (median 1.37%). Almost every row with a recorded size is 300x250, so size cannot explain much. Two groups lack a usable size: `0x0` rows (1.71% CTR) and rows with the size missing entirely (3,314 rows, 5.59% CTR). Both are data-recording gaps rather than design findings, but they show the missing-size flag is not random, so I would not just drop it.

## 8. Strongest campaigns and glitch source

In [11]:
display(Q['q11_best_campaigns']); display(Q['q12_glitch_rows_by_platform'])

,campaign_item_id,advertiser_name,days_,impressions,ctr_pct
0,3056,Maharashtra,135,2116590,8.085
1,3050,Kuala Lumpur,145,3426359,7.293
2,3104,North,40,2879003,6.898
3,3050,Karnataka,141,1571088,6.526
4,3054,Karnataka,450,2080093,5.514
5,3055,Tamil Nadu,116,1159335,5.324
6,3107,Thailand,129,1293465,5.018
7,3133,Orissa,438,1419422,4.391
8,3130,Thailand,370,1726542,4.186
9,3134,Orissa,331,1161729,4.114


,platform,glitch_rows
0,Google Ads,58
1,DV360,50
2,Facebook Ads,49


## 9. Correlation heatmap (numeric columns)

In [12]:
num = v[['no_of_days','campaign_budget_usd','impressions','clicks','media_cost_usd','ctr']]
plt.figure(figsize=(6,4.5)); sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='crest', vmin=-1, vmax=1); plt.title('Correlation of numeric columns'); plt.tight_layout(); plt.savefig('charts/07_correlation_heatmap.png', dpi=150); plt.show()

`media_cost_usd` correlates strongly with clicks (0.85) and impressions (0.93) because it is *an outcome of the campaign*. That is why my predictor excludes it: using it would be data leakage.